# 使用 Weaviate 与 OpenAI 向量化模块进行嵌入搜索

本 notebook 适用于以下场景：
* 你的数据尚未向量化
* 你希望在数据上运行向量搜索
* 你希望将 Weaviate 与 OpenAI 模块（[text2vec-openai](https://weaviate.io/developers/weaviate/modules/retriever-vectorizer-modules/text2vec-openai)）结合使用，由它为你生成向量嵌入。

本 notebook 将带你完成一个简单的流程：设置 Weaviate 实例、（使用 OpenAI API 密钥）连接到该实例、配置数据 schema、导入数据（这会自动为你的数据生成向量嵌入），以及运行语义搜索。

对于希望在安全环境中将我们的嵌入与自有数据一起存储和搜索、以支持聊天机器人、主题建模等生产用例的客户来说，这是一个常见需求。

## 什么是 Weaviate

Weaviate 是一个开源向量搜索引擎，它将数据对象与其向量一起存储。这使得向量搜索可以与结构化过滤结合使用。

Weaviate 使用 KNN 算法创建针对向量优化的索引，让你的查询运行得极快。可在[这里](https://weaviate.io/blog/why-is-vector-search-so-fast)了解更多。

Weaviate 让你可以使用自己喜欢的机器学习模型，并无缝扩展到数十亿个数据对象。

### 部署选项

无论你的场景或生产环境如何，Weaviate 都有适合你的选项。你可以通过以下方式部署 Weaviate：
* 自托管——你可以在本地使用 docker 部署 Weaviate，或部署在任何你想要的服务器上。
* SaaS——你可以使用 [Weaviate Cloud Service (WCS)](https://console.weaviate.io/) 托管你的 Weaviate 实例。
* 混合 SaaS——你可以在自己的私有云服务中部署 Weaviate。

### 编程语言

Weaviate 提供四种[客户端库](https://weaviate.io/developers/weaviate/client-libraries)，让你可以从应用中与其通信：
* [Python](https://weaviate.io/developers/weaviate/client-libraries/python)
* [JavaScript](https://weaviate.io/developers/weaviate/client-libraries/javascript)
* [Java](https://weaviate.io/developers/weaviate/client-libraries/java)
* [Go](https://weaviate.io/developers/weaviate/client-libraries/go)

此外，Weaviate 还有一个 [REST 层](https://weaviate.io/developers/weaviate/api/rest/objects)。基本上，你可以从任何支持 REST 请求的语言调用 Weaviate。

## 演示流程
演示流程如下：
- **前提条件设置**：创建 Weaviate 实例并安装所需的库
- **连接**：连接到你的 Weaviate 实例 
- **Schema 配置**：配置数据的 schema
    - *注意*：可以在这里定义使用哪个 OpenAI 嵌入模型
    - *注意*：可以在这里配置要索引哪些属性
- **导入数据**：加载演示数据集并将其导入 Weaviate
    - *注意*：导入过程会根据 schema 中的配置自动为你的数据建立索引
    - *注意*：你无需显式地向量化数据，Weaviate 会与 OpenAI 通信替你完成
- **运行查询**：执行查询 
    - *注意*：你无需显式地向量化查询，Weaviate 会与 OpenAI 通信替你完成

完成本 notebook 后，你应该对如何设置和使用向量数据库有了基本的了解，并可以继续探索利用我们的嵌入的更复杂用例。

## Weaviate 中的 OpenAI 模块
所有 Weaviate 实例都配备了 [text2vec-openai](https://weaviate.io/developers/weaviate/modules/retriever-vectorizer-modules/text2vec-openai) 模块。

该模块负责在导入（或任何 CRUD 操作）期间以及运行查询时处理向量化。

### 无需手动向量化数据
这对你来说是个好消息。借助 [text2vec-openai](https://weaviate.io/developers/weaviate/modules/retriever-vectorizer-modules/text2vec-openai)，你无需手动向量化数据，因为 Weaviate 会在必要时替你调用 OpenAI。

你只需要：
1. 在连接 Weaviate 客户端时提供你的 OpenAI API 密钥
2. 在 Schema 中定义要使用哪个 OpenAI 向量化器

## 前提条件

开始这个项目之前，我们需要完成以下设置：

* 创建一个 `Weaviate` 实例
* 安装库
    * `weaviate-client`
    * `datasets`
    * `apache-beam`
* 获取你的 [OpenAI API 密钥](https://beta.openai.com/account/api-keys)

===========================================================
### 创建 Weaviate 实例

创建 Weaviate 实例有 2 种选择：

1. （推荐）[Weaviate Cloud Service](https://console.weaviate.io/)——在云端托管你的 Weaviate 实例。免费沙盒对于本 cookbook 来说绰绰有余。
2. 使用 Docker 在本地安装并运行 Weaviate。

#### 选项 1——WCS 安装步骤

使用 [Weaviate Cloud Service](https://console.weaviate.io/)（WCS）创建一个免费的 Weaviate 集群。
1. 创建免费账户和/或登录 [WCS](https://console.weaviate.io/)
2. 使用以下设置创建一个 `Weaviate Cluster`：
    * Sandbox：`Sandbox Free`
    * Weaviate Version：使用默认值（最新版）
    * OIDC Authentication：`Disabled`
3. 你的实例应在一两分钟内就绪
4. 记下 `Cluster Id`。该链接会带你进入集群的完整路径（稍后连接时需要用到）。它应类似于：`https://your-project-name.weaviate.network` 

#### 选项 2——使用 Docker 的本地 Weaviate 实例

使用 Docker 在本地安装并运行 Weaviate。
1. 下载 [./docker-compose.yml](./docker-compose.yml) 文件
2. 然后打开终端，进入 docker-compose.yml 文件所在的目录，使用以下命令启动 docker：`docker-compose up -d`
3. 就绪后，你的实例应可通过 [http://localhost:8080](http://localhost:8080) 访问

注意：要关闭 docker 实例，可以运行：`docker-compose down`

##### 了解更多
要进一步了解如何在 Docker 中使用 Weaviate，请参阅[安装文档](https://weaviate.io/developers/weaviate/installation/docker-compose)。

===========================================================    
## 安装所需的库

运行本项目之前，请确保已安装以下库：

### Weaviate Python 客户端

[Weaviate Python 客户端](https://weaviate.io/developers/weaviate/client-libraries/python)让你可以从 Python 项目中与 Weaviate 实例通信。

### datasets 与 apache-beam

要加载示例数据，你需要 `datasets` 库及其依赖 `apache-beam`。

In [ ]:
# Install the Weaviate client for Python
!pip install weaviate-client>=3.11.0

# Install datasets and apache-beam to load the sample datasets
!pip install datasets apache-beam

===========================================================
## 准备 OpenAI API 密钥

`OpenAI API key` 用于在导入时向量化你的数据，以及运行查询。

如果你还没有 OpenAI API 密钥，可以从 [https://beta.openai.com/account/api-keys](https://beta.openai.com/account/api-keys) 获取。

获取密钥后，请将其以 `OPENAI_API_KEY` 的名称添加到环境变量中。

In [ ]:
# Export OpenAI API Key
!export OPENAI_API_KEY="your key"

In [ ]:
# Test that your OpenAI API key is correctly set as an environment variable
# Note. if you run this notebook locally, you will need to reload your terminal and the notebook for the env variables to be live.
import os

# Note. alternatively you can set a temporary env variable like this:
# os.environ["OPENAI_API_KEY"] = 'your-key-goes-here'

if os.getenv("OPENAI_API_KEY") is not None:
    print ("OPENAI_API_KEY is ready")
else:
    print ("OPENAI_API_KEY environment variable not found")

## 连接到你的 Weaviate 实例

在本节中，我们将：

1. 测试环境变量 `OPENAI_API_KEY`——**请确保**你已完成 [#准备 OpenAI API 密钥](#准备-openai-api-密钥) 中的步骤
2. 使用你的 `OpenAI API Key` 连接到 Weaviate
3. 并测试客户端连接

### 客户端 

完成这一步后，`client` 对象将用于执行所有与 Weaviate 相关的操作。

In [ ]:
import weaviate
from datasets import load_dataset
import os

# Connect to your Weaviate instance
client = weaviate.Client(
    url="https://your-wcs-instance-name.weaviate.network/",
    # url="http://localhost:8080/",
    auth_client_secret=weaviate.auth.AuthApiKey(api_key="<YOUR-WEAVIATE-API-KEY>"), # comment out this line if you are not using authentication for your Weaviate instance (i.e. for locally deployed instances)
    additional_headers={
        "X-OpenAI-Api-Key": os.getenv("OPENAI_API_KEY")
    }
)

# Check if your instance is live and ready
# This should return `True`
client.is_ready()

# Schema

在本节中，我们将：
1. 为你的数据配置数据 schema
2. 选择 OpenAI 模块

> 这是第二步，也是最后一步需要 OpenAI 特定配置的步骤。
> 完成这一步后，其余的说明将只涉及 Weaviate，因为 OpenAI 相关的任务将被自动处理。


## 什么是 schema

在 Weaviate 中，你需要创建 __schema__ 来描述要搜索的每一种实体。

schema 是你告诉 Weaviate 以下信息的方式：
* 应使用哪个嵌入模型来向量化数据
* 你的数据由什么组成（属性名称和类型）
* 哪些属性应被向量化并建立索引

在本 cookbook 中，我们将使用一个 `Articles` 数据集，其中包含：
* `title`
* `content`
* `url`

我们希望向量化 `title` 和 `content`，但不向量化 `url`。

为了向量化和查询数据，我们将使用 `text-embedding-3-small`。

In [ ]:
# Clear up the schema, so that we can recreate it
client.schema.delete_all()
client.schema.get()

# Define the Schema object to use `text-embedding-3-small` on `title` and `content`, but skip it for `url`
article_schema = {
    "class": "Article",
    "description": "A collection of articles",
    "vectorizer": "text2vec-openai",
    "moduleConfig": {
        "text2vec-openai": {
          "model": "ada",
          "modelVersion": "002",
          "type": "text"
        }
    },
    "properties": [{
        "name": "title",
        "description": "Title of the article",
        "dataType": ["string"]
    },
    {
        "name": "content",
        "description": "Contents of the article",
        "dataType": ["text"]
    },
    {
        "name": "url",
        "description": "URL to the article",
        "dataType": ["string"],
        "moduleConfig": { "text2vec-openai": { "skip": True } }
    }]
}

# add the Article schema
client.schema.create_class(article_schema)

# get the schema to make sure it worked
client.schema.get()

## 导入数据

在本节中，我们将：
1. 加载 Simple Wikipedia 数据集
2. 配置 Weaviate 批量导入（以提高导入效率）
3. 将数据导入 Weaviate

> 注意：<br/>
> 如前所述，我们无需手动向量化数据。<br/>
> [text2vec-openai](https://weaviate.io/developers/weaviate/modules/retriever-vectorizer-modules/text2vec-openai) 模块会处理这一切。

In [ ]:
### STEP 1 - load the dataset

from datasets import load_dataset
from typing import List, Iterator

# We'll use the datasets library to pull the Simple Wikipedia dataset for embedding
dataset = list(load_dataset("wikipedia", "20220301.simple")["train"])

# For testing, limited to 2.5k articles for demo purposes
dataset = dataset[:2_500]

# Limited to 25k articles for larger demo purposes
# dataset = dataset[:25_000]

# for free OpenAI acounts, you can use 50 objects
# dataset = dataset[:50]

In [ ]:
### Step 2 - configure Weaviate Batch, with
# - starting batch size of 100
# - dynamically increase/decrease based on performance
# - add timeout retries if something goes wrong

client.batch.configure(
    batch_size=10, 
    dynamic=True,
    timeout_retries=3,
#   callback=None,
)

In [ ]:
### Step 3 - import data

print("Importing Articles")

counter=0

with client.batch as batch:
    for article in dataset:
        if (counter %10 == 0):
            print(f"Import {counter} / {len(dataset)} ")

        properties = {
            "title": article["title"],
            "content": article["text"],
            "url": article["url"]
        }
        
        batch.add_data_object(properties, "Article")
        counter = counter+1

print("Importing Articles complete")       

In [ ]:
# Test that all data has loaded – get object count
result = (
    client.query.aggregate("Article")
    .with_fields("meta { count }")
    .do()
)
print("Object count: ", result["data"]["Aggregate"]["Article"], "\n")

In [ ]:
# Test one article has worked by checking one object
test_article = (
    client.query
    .get("Article", ["title", "url", "content"])
    .with_limit(1)
    .do()
)["data"]["Get"]["Article"][0]

print(test_article['title'])
print(test_article['url'])
print(test_article['content'])

### 搜索数据

和上面一样，我们将向新索引发送一些查询，并根据与现有向量的接近程度获取结果

In [ ]:
def query_weaviate(query, collection_name):
    
    nearText = {
        "concepts": [query],
        "distance": 0.7,
    }

    properties = [
        "title", "content", "url",
        "_additional {certainty distance}"
    ]

    result = (
        client.query
        .get(collection_name, properties)
        .with_near_text(nearText)
        .with_limit(10)
        .do()
    )
    
    # Check for errors
    if ("errors" in result):
        print ("\033[91mYou probably have run out of OpenAI API calls for the current minute – the limit is set at 60 per minute.")
        raise Exception(result["errors"][0]['message'])
    
    return result["data"]["Get"][collection_name]

In [ ]:
query_result = query_weaviate("modern art in Europe", "Article")

for i, article in enumerate(query_result):
    print(f"{i+1}. { article['title']} (Score: {round(article['_additional']['certainty'],3) })")

In [ ]:
query_result = query_weaviate("Famous battles in Scottish history", "Article")

for i, article in enumerate(query_result):
    print(f"{i+1}. { article['title']} (Score: {round(article['_additional']['certainty'],3) })")

感谢你的跟随学习，现在你已经具备了设置自己的向量数据库、并利用嵌入实现各种酷炫功能的能力——祝你玩得开心！对于更复杂的用例，请继续学习本仓库中的其他 cookbook 示例。